In [7]:
from google.colab import output
output.enable_custom_widget_manager()

In [8]:

# Round Robin CPU Scheduling Simulator
# Operating Systems Assignment 1

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from collections import deque
import pandas as pd

# ---------------- INPUT CONTROLS ----------------

activity_input = widgets.Dropdown(
    options=[
        ("Activity 1 - No Context Switch Cost", "activity1"),
        ("Activity 2 - 1 ms Context Switch Cost", "activity2")
    ],
    value="activity1",
    description="Activity:"
)

process_input = widgets.Text(
    value="5,3,4,2",
    description="Burst times:",
    placeholder="Example: 5,3,4,2"
)

arrival_input = widgets.Text(
    value="0,1,2,3",
    description="Arrival times:"
)

quantum_input = widgets.IntSlider(
    value=2,
    min=1,
    max=10,
    step=1,
    description="Time quantum:"
)

context_input = widgets.IntSlider(
    value=0,
    min=0,
    max=5,
    step=1,
    description="Switch cost:"
)

run_button = widgets.Button(
    description="Run Simulation",
    button_style="success",
    icon="play"
)

play_button = widgets.Play(
    value=0,
    min=0,
    max=1,
    step=1,
    interval=800
)

step_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=1,
    step=1,
    description="Step:",
    continuous_update=False
)

next_button = widgets.Button(
    description="Next Step",
    button_style="info",
    icon="step-forward"
)

reset_button = widgets.Button(
    description="Reset",
    button_style="warning",
    icon="refresh"
)

chart_output = widgets.Output()
result_output = widgets.Output()
status_output = widgets.Output()

schedule = []
results = []
total_time = 0


# ---------------- ACTIVITY SETTINGS ----------------

def load_activity(change=None):
    if activity_input.value == "activity1":
        process_input.value = "5,3,4,2"
        arrival_input.value = "0,1,2,3"
        quantum_input.value = 2
        context_input.value = 0
    else:
        process_input.value = "5,4,3,2"
        arrival_input.value = "0,1,2,3"
        quantum_input.value = 2
        context_input.value = 1

activity_input.observe(load_activity, names="value")


# ---------------- ROUND ROBIN ALGORITHM ----------------

def make_schedule(bursts, arrivals, quantum, switch_cost):
    remaining = bursts.copy()
    n = len(bursts)

    first_start = [None] * n
    completion = [0] * n
    segments = []
    ready = deque()
    added = [False] * n

    time = 0
    completed = 0
    last_process = None
    context_switches = 0

    while completed < n:

        # Add processes that have arrived
        for i in range(n):
            if arrivals[i] <= time and not added[i]:
                ready.append(i)
                added[i] = True

        # Move to the next arrival if no process is ready
        if not ready:
            future = [
                arrivals[i] for i in range(n)
                if not added[i]
            ]

            if not future:
                break

            time = min(future)
            continue

        p = ready.popleft()

        # Add context-switch overhead
        if (
            last_process is not None
            and last_process != p
            and switch_cost > 0
        ):
            segments.append({
                "process": "CS",
                "start": time,
                "end": time + switch_cost
            })

            context_switches += 1
            time += switch_cost

            # Add processes that arrived during the switch
            for i in range(n):
                if arrivals[i] <= time and not added[i]:
                    ready.append(i)
                    added[i] = True

        if first_start[p] is None:
            first_start[p] = time

        start = time
        run_time = min(quantum, remaining[p])

        time += run_time
        remaining[p] -= run_time

        segments.append({
            "process": f"P{p + 1}",
            "start": start,
            "end": time
        })

        # Add new arrivals to the ready queue
        for i in range(n):
            if arrivals[i] <= time and not added[i]:
                ready.append(i)
                added[i] = True

        if remaining[p] > 0:
            ready.append(p)
        else:
            completion[p] = time
            completed += 1

        last_process = p

    # Calculate scheduling metrics
    process_results = []

    for i in range(n):
        tat = completion[i] - arrivals[i]
        wt = tat - bursts[i]
        rt = first_start[i] - arrivals[i]

        process_results.append({
            "Process": f"P{i + 1}",
            "AT": arrivals[i],
            "BT": bursts[i],
            "CT": completion[i],
            "TAT": tat,
            "WT": wt,
            "RT": rt
        })

    return segments, process_results, context_switches


# ---------------- DRAW GANTT CHART ----------------

def draw_chart(step):
    with chart_output:
        clear_output(wait=True)

        if not schedule:
            print("Click Run Simulation first.")
            return

        fig, ax = plt.subplots(figsize=(11, 4))

        visible = schedule[:step]

        names = sorted(
            set(
                s["process"] for s in schedule
                if s["process"] != "CS"
            ),
            key=lambda name: int(name[1:])
        )

        rows = {name: i for i, name in enumerate(names)}

        if any(s["process"] == "CS" for s in schedule):
            rows["CS"] = len(names)

        for seg in visible:
            name = seg["process"]
            start = seg["start"]
            width = seg["end"] - seg["start"]
            y = rows[name]

            ax.barh(
                y,
                width,
                left=start,
                height=0.55,
                edgecolor="black",
                alpha=0.85
            )

            ax.text(
                start + width / 2,
                y,
                name,
                ha="center",
                va="center",
                fontsize=9
            )

        current_time = (
            visible[-1]["end"] if visible else 0
        )

        ax.axvline(
            current_time,
            linestyle="--",
            linewidth=2,
            label=f"Time = {current_time}"
        )

        ax.set_yticks(list(rows.values()))
        ax.set_yticklabels(list(rows.keys()))
        ax.set_xlabel("CPU Time (ms)")
        ax.set_ylabel("Process")
        ax.set_title("Round Robin Gantt Chart")
        ax.set_xlim(0, max(total_time, 1))
        ax.set_xticks(range(int(total_time) + 1))
        ax.grid(axis="x", alpha=0.3)
        ax.legend()

        plt.tight_layout()
        plt.show()
        plt.close(fig)

        print(
            f"Current time: {current_time} ms | "
            f"Steps: {step}/{len(schedule)}"
        )


# ---------------- ANIMATION CONTROLS ----------------

# Make the buttons easier to see
display(HTML("""
<style>
.jupyter-widgets button,
.widget-button {
    background-color: #1976D2 !important;
    color: white !important;
    border: 1px solid #125699 !important;
    border-radius: 6px !important;
    min-height: 35px !important;
    padding: 5px 12px !important;
}
</style>
"""))

def update_step(change):
    if schedule:
        draw_chart(change["new"])

step_slider.observe(update_step, names="value")


def play_changed(change):
    if schedule and step_slider.value != change["new"]:
        step_slider.value = change["new"]

play_button.observe(play_changed, names="value")


def next_step_clicked(button):
    if schedule:
        step_slider.value = min(
            step_slider.value + 1,
            len(schedule)
        )

next_button.on_click(next_step_clicked)


def reset_clicked(button):
    play_button.value = 0
    step_slider.value = 0
    draw_chart(0)

reset_button.on_click(reset_clicked)

# ---------------- RUN SIMULATION ----------------

def run_clicked(button):
    global schedule, results, total_time

    try:
        bursts = [
            int(x.strip())
            for x in process_input.value.split(",")
        ]

        arrivals = [
            int(x.strip())
            for x in arrival_input.value.split(",")
        ]

        if not bursts or any(x <= 0 for x in bursts):
            raise ValueError(
                "Burst times must be positive integers."
            )

        if len(bursts) != len(arrivals):
            raise ValueError(
                "Enter one arrival time per process."
            )

        if any(x < 0 for x in arrivals):
            raise ValueError(
                "Arrival times cannot be negative."
            )

        quantum = quantum_input.value
        switch_cost = context_input.value

        schedule, results, switches = make_schedule(
            bursts,
            arrivals,
            quantum,
            switch_cost
        )

        total_time = max(
            (s["end"] for s in schedule),
            default=0
        )

        play_button.value = 0
        step_slider.max = len(schedule)
        step_slider.value = 0
        play_button.max = len(schedule)

        df = pd.DataFrame(results)

        busy_time = sum(bursts)
        overhead = switches * switch_cost

        efficiency = (
            busy_time / total_time * 100
            if total_time > 0 else 0
        )

        with result_output:
            clear_output(wait=True)

            print("PROCESS SCHEDULING RESULTS")
            display(df)

            print(
                "\nAverage Waiting Time:",
                round(df["WT"].mean(), 2), "ms"
            )

            print(
                "Average Turnaround Time:",
                round(df["TAT"].mean(), 2), "ms"
            )

            print(
                "Average Response Time:",
                round(df["RT"].mean(), 2), "ms"
            )

            print("Context Switches:", switches)
            print("Total Switch Overhead:", overhead, "ms")
            print("Total Elapsed Time:", total_time, "ms")
            print(
                "CPU Efficiency:",
                round(efficiency, 2), "%"
            )

        with status_output:
            clear_output(wait=True)
            print(
                "Simulation ready! Use Play or Next Step."
            )

        draw_chart(0)

    except Exception as error:
        with status_output:
            clear_output(wait=True)
            print("Input error:", error)

run_button.on_click(run_clicked)


# ---------------- DISPLAY SIMULATOR ----------------

display(widgets.VBox([
    widgets.HTML(
        "<h3>Round Robin CPU Scheduling Simulator</h3>"
    ),
    activity_input,
    process_input,
    arrival_input,
    quantum_input,
    context_input,
    run_button,
    widgets.HTML("<b>Animation Controls</b>"),
    widgets.HBox([
        play_button,
        next_button,
        reset_button
    ]),
    step_slider,
    status_output,
    chart_output,
    result_output
]))


In [9]:
!pip install -q ipywidgets